Model

In [7]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, n_features=None, class_weights=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.class_weights = class_weights
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))

        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            thresholds = np.unique(X_col)

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left = y[X_col < threshold]
        right = y[X_col >= threshold]

        if len(left) == 0 or len(right) == 0:
            return float("inf")

        def weighted_gini(group):
            if len(group) == 0:
                return 0

            classes, counts = np.unique(group, return_counts=True)

            if self.class_weights is not None:
                weighted_counts = np.array([
                    counts[i] * self.class_weights.get(classes[i], 1)
                    for i in range(len(classes))
                ])
            else:
                weighted_counts = counts

            probs = weighted_counts / np.sum(weighted_counts)
            return 1 - np.sum(probs ** 2)

        n = len(y)
        return (len(left)/n)*weighted_gini(left) + (len(right)/n)*weighted_gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)

        if self.class_weights is not None:
            weighted_counts = np.array([
                counts[i] * self.class_weights.get(classes[i], 1)
                for i in range(len(classes))
            ])
        else:
            weighted_counts = counts

        return classes[np.argmax(weighted_counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

Metrics

In [8]:
def compute_metrics_multiclass(y_true, y_pred):
    classes = np.unique(y_true)
    accuracy = np.sum(y_true == y_pred) / len(y_true)

    precision_list = []
    recall_list = []
    f1_list = []

    for c in classes:
        TP = np.sum((y_true == c) & (y_pred == c))
        FP = np.sum((y_true != c) & (y_pred == c))
        FN = np.sum((y_true == c) & (y_pred != c))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    return accuracy, np.mean(precision_list), np.mean(recall_list), np.mean(f1_list)

def confusion_matrix_custom(y_true, y_pred):
    classes = np.unique(y_true)
    n = len(classes)
    matrix = np.zeros((n, n), dtype=int)

    for i, c1 in enumerate(classes):
        for j, c2 in enumerate(classes):
            matrix[i, j] = np.sum((y_true == c1) & (y_pred == c2))

    return matrix, classes

In [9]:
from preprocessing import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten")

Loading MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 18s 2us/step
Split completed: Train=54000, Val=6000, Test=10000


Training and Testing

In [10]:
import time

print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(max_depth=10, min_samples=10, n_features=50)

print("Training...")
start = time.time()
tree.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.2f}s")

print("\nEvaluating...")
preds = tree.predict(X_val)

acc, prec, rec, f1 = compute_metrics_multiclass(y_val, preds)

print("\n" + "="*50)
print("   DECISION TREE PERFORMANCE")
print("="*50)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

cm, classes = confusion_matrix_custom(y_val, preds)

print("\nConfusion Matrix:")
print("Rows = Actual, Columns = Predicted")
print(cm)

Initializing Custom Decision Tree...
Training...
Training time: 5.45s

Evaluating...

   DECISION TREE PERFORMANCE
Accuracy : 0.9835
Precision: 0.9600
Recall   : 0.9453
F1-score : 0.9525

Confusion Matrix:
Rows = Actual, Columns = Predicted
[[ 527   60]
 [  39 5374]]
